# 1 · Ask Your Data: Vibe Analytics

<a href="https://colab.research.google.com/github/FH-Prevail/ailab/blob/main/notebooks/1_ask_your_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

*AI Lab · Vibe Analytics, Vibe Coding And MCP · Sina Mirshahi, FH OÖ*


**Block 1, hands-on 1.** You are the logistics analyst of a company in Upper Austria that sells spare parts. The company must sign new contracts with its carriers for 2027. The head of logistics asks:

> **For each country, which delivery company (carrier) should we use? And when is the faster, more expensive Express service worth its price?**

You have one export (a file from the company's system) with every shipment of 2025. Your AI writes the code. Your job: say exactly what you mean, check what the AI did, and decide what the numbers really show.


**How to work**

1. In Colab choose **File, Save a copy in Drive**. Your copy keeps your changes.
2. Work in pairs: one person types, the other compares every answer with the `# CHECK` values. Change roles after each section.
3. Run cells with **Shift + Enter**.

| Label | Meaning |
|---|---|
| **Ask your AI** | a prompt to paste into Gemini in Colab, AI Studio, ChatGPT, Claude or Copilot |
| `# paste the AI's code here` | an empty cell for the code your AI wrote |
| `# CHECK` | the number you should see with the course data |
| **Our answer** | a closed cell with our code: open it (click *Show code*) after you have tried |
| **Your answer** | a few words of your own: double-click, write, Shift + Enter |

The data is not real (it is synthetic): we invented the company and its carriers. Share only course data with AI tools, never company data.

### Words we use

| Word | Meaning |
|---|---|
| **carrier** | A delivery company that transports our shipments. |
| **Standard**, **Express** | Normal and fast delivery; Express costs more. |
| **on time** | Delivered on or before the promised day. |
| **known result** | We know if the shipment was on time or late (a lost shipment counts as late). |
| **not known yet** | The shipment is still on its way, and its promised day is still in the future. |
| **metric contract** | Our written rules for counting, so that everybody gets the same number. |
| **vibe analytics** | You ask an AI in plain words to analyse data, and it writes and runs the code. |
| **vibe coding** | You ask an AI in plain words to build an app, and you test what it built. |
| **MCP** | Model Context Protocol: a standard way to connect an AI to tools. |
| **host** | The app you talk to, for example Claude or ChatGPT; it passes messages between you, the model and the tools. |
| **model** | The AI that reads your question and writes the answer, for example Gemini. |
| **tool** | A small program on an MCP server that the AI can use, for example to count shipments. |

In [ ]:
# Start here: run this cell first. It loads the export of all 2025 shipments.
from pathlib import Path
import pandas as pd

URL = "https://raw.githubusercontent.com/FH-Prevail/ailab/main/data/shipments_2025.csv"
if Path("data/shipments_2025.csv").exists():          # a copy next to the notebook is used first
    URL = "data/shipments_2025.csv"
raw = pd.read_csv(URL, parse_dates=["ship_date", "promised_date", "delivered_date"])   # the export: never changed
df = raw.copy()                                                                         # the copy you and your AI work on
print(f"Ready: {len(df):,} rows, {df.shape[1]} columns, ship dates {df['ship_date'].min():%Y-%m-%d} to {df['ship_date'].max():%Y-%m-%d}")
df.head()
# CHECK: Ready: 6,043 rows, 16 columns, ship dates 2025-01-02 to 2025-12-31

### Which AI?

- **Gemini in Colab:** click the Gemini button or *Generate* in an empty cell. It sees `df` (the table) and writes code into the notebook.
- **Any chat AI in another tab** (AI Studio, ChatGPT, Claude, Copilot): paste the context below first. Then paste the code it gives you into the empty cell and run it here. If your AI runs code itself, you may upload [shipments_2025.csv](https://raw.githubusercontent.com/FH-Prevail/ailab/main/data/shipments_2025.csv) there instead.

> **Context for a chat AI:** *"I work in Google Colab with pandas. My table `df` is the export of our 2025 shipments. Columns: shipment_id, ship_date, promised_date, delivered_date (empty if not delivered), status (delivered, lost or in_transit), carrier, service (Standard or Express), warehouse (Linz or Wels), country (AT, DE, CZ, HU, IT, SI), city, distance_km, weight_kg, parcels, cost_eur, damaged (1 = damaged), customer_id. The export ran at the end of 31 December 2025. Do not change df itself; work on a copy. Write short pandas code with comments."*

## 1. The quick question

**Ask your AI** exactly this, nothing more:

```
Which carrier in df is the most reliable?
```

Run its code below and note the answer: which carrier, which number, and how did it define "reliable"?

In [ ]:
df = raw.copy()   # start again from the untouched export
# paste the AI's code here

**Your answer:** carrier, number, and the definition your AI used.

## 2. Check the data first

**Ask your AI:**

```
Before any analysis, check the table df for data quality problems: repeated rows,
different spellings of the same name, impossible or extreme values, and missing
values. For each problem give the number of rows, one example and the evidence.
Do not fix anything yet.
```

In [ ]:
df = raw.copy()   # start again from the untouched export
# paste the AI's code here

**Discuss with your partner:** which problems did your AI *show* with evidence, which did it only *suspect*, and what would confirm them? Then open our check.

In [ ]:
#@title Our answer: four problems in the export (open after you tried)
repeats = raw[raw.duplicated("shipment_id")]
print("1. Repeated rows:", len(repeats), "rows repeat a shipment_id;", (repeats["carrier"] == "Luchs Logistik").sum(), "of them are Luchs Logistik")
print("2. Carrier names:", raw["carrier"].nunique(), "spellings for four carriers:", sorted(raw["carrier"].unique()))
wels = raw[raw["warehouse"] == "Wels"]
before = wels[wels["ship_date"] < "2025-09-01"]["weight_kg"].median()
after = wels[wels["ship_date"] >= "2025-09-01"]["weight_kg"].median()
print(f"3. Weight at Wels: median {before:,.1f} before September, {after:,.1f} from September (a new scale writes grams)")
missing = raw[raw["delivered_date"].isna()]
print("4. No delivery date:", len(missing), "rows:", missing["status"].value_counts().to_dict())
# CHECK: 1. Repeated rows: 125 rows repeat a shipment_id; 91 of them are Luchs Logistik
# CHECK: 2. Carrier names: 6 spellings for four carriers
# CHECK: 3. Weight at Wels: median 18.5 before September, 17,700.0 from September
# CHECK: 4. No delivery date: 71 rows: {'in_transit': 44, 'lost': 27}

## 3. Agree on a definition, then write a good prompt

"Reliable" and "on time" sound clear. They are not. Here are six real rows from the export. Calculate the on-time rate **by hand** with your partner:

| # | shipment_id | ship_date | promised_date | delivered_date | status | carrier |
|---|---|---|---|---|---|---|
| 1 | SH-25-05430 | 2025-12-01 | 2025-12-03 | 2025-12-03 | delivered | Luchs Logistik |
| 2 | SH-25-05842 | 2025-12-22 | 2025-12-24 | 2025-12-30 | delivered | Luchs Logistik |
| 3 | SH-25-05842 | 2025-12-22 | 2025-12-24 | 2025-12-30 | delivered | Luchs Logistik |
| 4 | SH-25-05850 | 2025-12-23 | 2025-12-30 | (empty) | lost | Biber Freight |
| 5 | SH-25-05809 | 2025-12-19 | 2025-12-24 | (empty) | in_transit | Biber Freight |
| 6 | SH-25-05918 | 2025-12-31 | 2026-01-05 | (empty) | in_transit | Luchs Logistik |

**Your answer:** the on-time rate of these six rows, and how you counted rows 3 to 6.

<details>
<summary><b>Three ways to count (open after you have tried)</b></summary>

- Count the repeated row only once. A lost shipment counts as late. A shipment still in transit after its promised date counts as late. A shipment whose promised date is still in the future is *not known yet*. Result: **1 of 4 = 25.0 %**.
- Remove the rows without a delivery date, and keep the repeated row: 1 of 3 = **33.3 %**.
- Count every row, and count every missing date as late: 1 of 6 = **16.7 %**.

</details>

Same six rows, three answers. **The definition decides the result.** These are our company's rules for counting, the *metric contract*:

1. One row per shipment: drop rows that repeat a shipment_id (keep the first).
2. Carrier names: trim spaces, drop ' GmbH' and write them in title case, which leaves four carriers.
3. Weight: from 1 September 2025 the new scale at Wels writes grams into weight_kg; divide those rows by 1,000.
4. On time: delivered on or before promised_date. Lost counts as late. Still in transit with the promised date already passed (on or before 31 December 2025) counts as late. Still in transit with the promised date after the export is not known yet: leave it out of the on-time rate and count it separately.
5. Transit days: calendar days from ship_date to delivered_date, delivered shipments only.
6. Damage rate: damaged shipments as a share of delivered shipments.
7. Cost per shipment: the average of cost_eur over all shipments.

**Ask your AI** (this is a good prompt: context, rules, question, output, proof):

```
Apply these rules to df, in this order, and print the number of rows after each rule:
1. Drop rows that repeat a shipment_id (keep the first).
2. Carrier names: trim spaces, remove " GmbH", title case.
3. Rows from warehouse Wels shipped on or after 2025-09-01 have weight in grams: divide weight_kg by 1000.
4. On time = delivered_date on or before promised_date. Lost counts as late. In transit with
   promised_date on or before 2025-12-31 counts as late. In transit with promised_date after
   2025-12-31 is not known yet: leave it out.
Then give a table per carrier: shipments, shipments with a known result, on time, on-time rate
in % with one decimal. Show the code you ran.
```

In [ ]:
df = raw.copy()   # start again from the untouched export
# paste the AI's code here

In [ ]:
#@title Our answer with the metric contract (open after you tried)
EXPORT = pd.Timestamp("2025-12-31")
clean = raw.drop_duplicates("shipment_id").copy()                                                  # rule 1
clean["carrier"] = clean["carrier"].str.strip().str.replace(r"\s+GmbH$", "", regex=True).str.title()  # rule 2
grams = (clean["warehouse"] == "Wels") & (clean["ship_date"] >= "2025-09-01")
clean.loc[grams, "weight_kg"] = clean.loc[grams, "weight_kg"] / 1000                                # rule 3
delivered = clean["status"] == "delivered"
overdue = (clean["status"] == "in_transit") & (clean["promised_date"] <= EXPORT)
clean["known"] = delivered | (clean["status"] == "lost") | overdue                                  # rule 4
clean["on_time"] = (delivered & (clean["delivered_date"] <= clean["promised_date"])).astype("Int64").where(clean["known"])
# on_time: 1 on time, 0 late or lost, empty = not known yet (the same as in the cleaned table)
print(f"{len(raw):,} rows -> {len(clean):,} shipments -> {clean['known'].sum():,} with a known result")

known = clean[clean["known"]]
table = known.groupby("carrier").agg(known=("on_time", "size"), on_time=("on_time", "sum"))
table["on_time_pct"] = (table["on_time"] / table["known"] * 100).round(1)
print(table.sort_values("on_time_pct", ascending=False))
# CHECK: 6,043 rows -> 5,918 shipments -> 5,881 with a known result
# CHECK: Falke Transport 90.8, Gams Cargo 88.9, Luchs Logistik 88.3, Biber Freight 85.2

**Compare** with your answer from section 1. What changed, and which rule changed it most? Here is what the quick question gives without any rule (a missing date counts as late):

In [ ]:
#@title The quick answer, without the rules
quick = (raw["delivered_date"] <= raw["promised_date"]).groupby(raw["carrier"]).mean() * 100
print(quick.round(1).sort_values(ascending=False))
# CHECK: Falke Transport 90.1, BIBER FREIGHT 89.9, Luchs Logistik 82.2

Two things to notice. First: "BIBER FREIGHT" looks like the second-best carrier. But these are only 79 rows, typed in capital letters in the summer. Second: removing the repeated rows alone moves Luchs Logistik from 82.2 % to 87.7 %, because its late shipments were in the export twice. The rule for missing results then gives 88.3 %.

> **Continue from here** if your cleaning did not work: the cell below loads the cleaned table, with the rules already applied. In this table, `on_time` is 1, 0 or empty (not known yet). So a simple average is the on-time rate.

In [ ]:
# Continue from here: the cleaned table with the rules applied (run it if your cleaning went wrong)
clean = pd.read_csv("https://raw.githubusercontent.com/FH-Prevail/ailab/main/data/shipments_2025_clean.csv", parse_dates=["ship_date", "promised_date", "delivered_date"])
clean["known"] = clean["on_time"].notna()             # on_time: 1 on time, 0 late or lost, empty = not known yet
known = clean[clean["known"]].copy()
delivered = clean["status"].eq("delivered")
print((known.groupby("carrier")["on_time"].mean() * 100).round(1))
# CHECK: Biber Freight 85.2, Falke Transport 90.8, Gams Cargo 88.9, Luchs Logistik 88.3

## 4. Compare within each country

Gams Cargo looks worse than Falke Transport. **Ask your AI:**

```
Use the cleaned data from section 3 (or the table clean from the cell above, where on_time
is 1, 0 or empty). Show the on-time rate per carrier within each destination country, with
the number of shipments with a known result in each cell. Is Gams Cargo really less reliable
than Falke Transport?
```

In [ ]:
df = raw.copy()   # start again from the untouched export
# paste the AI's code here

In [ ]:
#@title Our answer: carrier by country (open after you tried)
rate = known.pivot_table(index="country", columns="carrier", values="on_time", aggfunc="mean") * 100
count = known.pivot_table(index="country", columns="carrier", values="on_time", aggfunc="size")
print(rate.round(1))
print(count)
print("Share of each carrier's shipments that go to Hungary, Italy or Slovenia:")
print((clean.groupby("carrier")["country"].apply(lambda c: c.isin(["HU", "IT", "SI"]).mean()) * 100).round(1))
# CHECK: AT: Gams 98.9, Falke 95.4; HU: Gams 88.5, Falke 76.5
# CHECK: IT: Gams 85.1, Falke 69.9; SI: Gams 88.4, Falke 82.7
# CHECK: Gams Cargo 83.5

Gams Cargo is better than Falke Transport **in every country where both deliver**. But overall, Gams is behind (88.9 % against 90.8 %). Why? 83.5 % of Gams shipments go to Hungary, Italy and Slovenia. These are long routes, and every carrier is late more often there. 65.9 % of Falke shipments stay in Austria. An overall average mixes the carrier with its routes. *A number can be correct and still answer the wrong question.*

## 5. If you have time: Express, December, damage

Choose one. **Ask your AI**, on the cleaned data (the table `clean` if you used the "continue from here" cell):

```
For each carrier compare Standard and Express: on-time rate, average transit days
(delivered shipments only) and average cost per shipment. When is Express worth its higher price?
```
```
Show the on-time rate per carrier and month. Does any carrier get much worse in some months?
```
```
Which carrier has the most damaged shipments? Show damaged shipments as a share of
delivered shipments, with the counts.
```

In [ ]:
df = raw.copy()   # start again from the untouched export
# paste the AI's code here

In [ ]:
#@title Our answers (open after you tried)
d = clean[delivered].assign(transit=lambda x: (x["delivered_date"] - x["ship_date"]).dt.days)
express = known.groupby(["carrier", "service"])["on_time"].mean().unstack() * 100
print("On time %:\n", express.round(1))
print("Transit days:\n", d.groupby(["carrier", "service"])["transit"].mean().unstack().round(2))
print("Cost per shipment:\n", clean.groupby(["carrier", "service"])["cost_eur"].mean().unstack().round(2))
print("December on time %:", (known[known["ship_date"].dt.month == 12].groupby("carrier")["on_time"].mean() * 100).round(1).to_dict())
print("Damaged %:", (d.groupby("carrier")["damaged"].mean() * 100).round(1).to_dict())
# CHECK: Luchs Logistik Express 42.3, Standard 91.4
# CHECK: Luchs Logistik transit Express 3.4, Standard 3.71
# CHECK: Luchs Logistik cost Express 42.11, Standard 25.71
# CHECK: December Luchs Logistik 64.0
# CHECK: Damaged Biber Freight 3.6

## 6. Your answer for the head of logistics

Write three sentences: your **recommendation** (which carrier for which country), the **number** that supports it, with its count, and **one limitation** (for example: these are observations from 2025, not proof for 2027; some groups have only a few shipments; what else you want to know before you sign a contract).

**Your answer:**

---
**Next:** open [notebook 2, Build The App](https://colab.research.google.com/github/FH-Prevail/ailab/blob/main/notebooks/2_build_the_app.ipynb) and write the spec (the description) of your app.